In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_6.py ──
"""Shared infrastructure for MLFP02 Exercise 6 — Logistic Regression and
Classification Foundations.

Contains: HDB data loading, binary target construction, design matrix
preparation, sigmoid implementation, the neg-log-likelihood + gradient used
by scipy.optimize, calibration-curve binning, and output-directory setup.

Technique-specific code (odds ratios, threshold sweeps, confusion matrices,
ROC/PR/calibration plots, ANOVA + Tukey HSD) does NOT belong here — it
lives in the per-technique files in solutions/ex_6/ and local/ex_6/.
"""

from pathlib import Path

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "mlfp02_ex6_logistic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS: list[str] = ["floor_area_sqm", "storey_mid", "remaining_lease"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale transactions (Singapore)
# ════════════════════════════════════════════════════════════════════════


def load_hdb_recent() -> pl.DataFrame:
    """Load HDB resale transactions filtered to 2020+.

    Returns a polars DataFrame with the raw columns from the dataset
    (resale_price, flat_type, floor_area_sqm, storey_range, lease_commence_date,
    month, etc.). No target construction — see build_classification_frame().
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def build_classification_frame(hdb_recent: pl.DataFrame) -> tuple[pl.DataFrame, float]:
    """Add the binary target and engineered features used by logistic regression.

    The target ``high_price`` = 1 if resale_price > median, 0 otherwise — this
    gives a balanced ~50/50 split so the baseline accuracy is 50%.

    Returns (dataframe, median_price).
    """
    median_price = hdb_recent["resale_price"].median()
    frame = hdb_recent.with_columns(
        (pl.col("resale_price") > median_price).cast(pl.Int32).alias("high_price"),
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2.0
        ).alias("storey_mid"),
        (
            99
            - (
                pl.col("month").str.to_date("%Y-%m").dt.year()
                - pl.col("lease_commence_date")
            )
        )
        .cast(pl.Float64)
        .alias("remaining_lease"),
    ).drop_nulls(
        subset=["floor_area_sqm", "storey_mid", "high_price", "remaining_lease"]
    )
    return frame, float(median_price)


def build_design_matrix(
    frame: pl.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]:
    """Standardise features and append an intercept column.

    Returns (X_with_intercept, y, X_mean, X_std, feature_names_with_intercept).
    """
    X_raw = frame.select(FEATURE_COLS).to_numpy().astype(np.float64)
    y = frame["high_price"].to_numpy().astype(np.float64)

    X_mean = X_raw.mean(axis=0)
    X_std = X_raw.std(axis=0)
    X_scaled = (X_raw - X_mean) / X_std

    n_obs = X_scaled.shape[0]
    X = np.column_stack([np.ones(n_obs), X_scaled])
    feature_names = ["intercept", *FEATURE_COLS]
    return X, y, X_mean, X_std, feature_names


# ════════════════════════════════════════════════════════════════════════
# SIGMOID + BERNOULLI LOG-LIKELIHOOD
# ════════════════════════════════════════════════════════════════════════


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid σ(z) = 1 / (1 + exp(-z)).

    For z ≥ 0 use 1/(1+exp(-z)); for z < 0 use exp(z)/(1+exp(z)). Both
    branches avoid overflow at the extremes.
    """
    z = np.asarray(z, dtype=np.float64)
    result = np.zeros_like(z)
    pos = z >= 0
    neg = ~pos
    result[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    exp_z = np.exp(z[neg])
    result[neg] = exp_z / (1.0 + exp_z)
    return result


def neg_log_likelihood_logistic(
    beta: np.ndarray, X: np.ndarray, y: np.ndarray
) -> float:
    """Negative Bernoulli log-likelihood: -Σ[y log p + (1-y) log(1-p)]."""
    z = X @ beta
    p = sigmoid(z)
    p = np.clip(p, 1e-15, 1 - 1e-15)
    ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float(-ll)


def neg_ll_gradient(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Gradient of the negative log-likelihood: -X^T (y - p)."""
    z = X @ beta
    p = sigmoid(z)
    return -X.T @ (y - p)


# ════════════════════════════════════════════════════════════════════════
# ORIGINAL-SCALE COEFFICIENT CONVERSION
# ════════════════════════════════════════════════════════════════════════


def unscale_coefficients(
    beta_scaled: np.ndarray, X_mean: np.ndarray, X_std: np.ndarray
) -> np.ndarray:
    """Convert coefficients fit on standardised features to the original scale.

    β_original[j]   = β_scaled[j] / σ_j  for j ≥ 1
    β_original[0]   = β_scaled[0] - Σ β_scaled[j] * μ_j / σ_j
    """
    beta_original = np.zeros_like(beta_scaled)
    beta_original[0] = beta_scaled[0] - float(np.sum(beta_scaled[1:] * X_mean / X_std))
    beta_original[1:] = beta_scaled[1:] / X_std
    return beta_original


# ════════════════════════════════════════════════════════════════════════
# CALIBRATION BINNING
# ════════════════════════════════════════════════════════════════════════


def calibration_bins(
    y: np.ndarray, p: np.ndarray, n_bins: int = 10
) -> tuple[list[float], list[float], list[int]]:
    """Equal-width binning over predicted probabilities.

    Returns (mean_predicted, mean_observed, counts) for non-empty bins only.
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    mean_pred: list[float] = []
    mean_obs: list[float] = []
    counts: list[int] = []
    for i in range(n_bins):
        hi = edges[i + 1] + (1e-12 if i == n_bins - 1 else 0.0)
        mask = (p >= edges[i]) & (p < hi)
        if mask.sum() > 0:
            mean_pred.append(float(p[mask].mean()))
            mean_obs.append(float(y[mask].mean()))
            counts.append(int(mask.sum()))
    return mean_pred, mean_obs, counts




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 6.3: Classification Metrics — Confusion Matrix,
#                         ROC, and Precision-Recall
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Compute and interpret the confusion matrix (TP, FP, TN, FN)
#   - Derive precision, recall, F1, and accuracy from the matrix
#   - Plot and interpret ROC curves + compute AUC
#   - Plot precision-recall curves for imbalanced-class awareness
#   - Apply classification metrics to bank mortgage pre-screening
#
# PREREQUISITES: Exercise 6.1 (logistic regression), 6.2 (thresholds)
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — the four quadrants of classification
#   2. Build — confusion matrix + derived metrics
#   3. Train — ROC curve + precision-recall curve
#   4. Visualise — ROC, PR, and annotated confusion matrix
#   5. Apply — bank mortgage pre-screening (illustrative S$ impact)
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_curve,
)



## THEORY — The Four Quadrants of Classification


In [ ]:
# Every binary classifier maps each observation into one of four cells:
#
#                      Predicted Positive    Predicted Negative
#   Actual Positive        TP                    FN
#   Actual Negative        FP                    TN
#
# From these four counts, we derive:
#   Accuracy  = (TP + TN) / N           — overall correctness
#   Precision = TP / (TP + FP)          — "of those I flagged, how many right?"
#   Recall    = TP / (TP + FN)          — "of those actually positive, how many caught?"
#   F1        = 2 * Prec * Rec / (P+R)  — harmonic mean of precision and recall
#
# Accuracy is misleading when classes are imbalanced. If 95% of
# transactions are normal, a model that always says "normal" gets 95%
# accuracy but catches ZERO anomalies. Precision and recall expose
# this failure.
#
# ROC curve plots TPR (recall) vs FPR across all thresholds. The area
# under the curve (AUC) measures discrimination ability irrespective of
# threshold choice.
#
# Precision-recall curve is more informative than ROC when the positive
# class is rare — it punishes models that achieve high recall only by
# drowning in false positives.



## TASK 2 — BUILD: fit model and compute confusion matrix


In [ ]:
print("\n" + "=" * 70)
print("  Classification Metrics — Confusion, ROC, Precision-Recall")
print("=" * 70)

# Load data and fit logistic regression
hdb_recent = load_hdb_recent()
frame, median_price = build_classification_frame(hdb_recent)
X, y, X_mean, X_std, feature_names = build_design_matrix(frame)
n_obs = X.shape[0]
n_positive = int(y.sum())

beta0 = np.zeros(X.shape[1])
result = minimize(
    neg_log_likelihood_logistic,
    beta0,
    args=(X, y),
    method="L-BFGS-B",
    jac=neg_ll_gradient,
    options={"maxiter": 1000, "ftol": 1e-12},
)
beta_scratch = result.x
p_scratch = sigmoid(X @ beta_scratch)

# Use a cost-optimal threshold (FN more costly than FP)
cost_fp, cost_fn = 30_000, 50_000
thresholds_sweep = np.linspace(0.1, 0.9, 81)
costs = []
for t in thresholds_sweep:
    cm_t = confusion_matrix(y, (p_scratch >= t).astype(int))
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    costs.append(fp_t * cost_fp + fn_t * cost_fn)
optimal_threshold = thresholds_sweep[np.argmin(costs)]

# Confusion matrix at optimal threshold
y_pred_opt = (p_scratch >= optimal_threshold).astype(int)
cm = confusion_matrix(y, y_pred_opt)
tn, fp, fn, tp = cm.ravel()

prec = precision_score(y, y_pred_opt)
rec = recall_score(y, y_pred_opt)
f1 = f1_score(y, y_pred_opt)
acc = accuracy_score(y, y_pred_opt)

print(f"\n=== Classification Metrics (threshold={optimal_threshold:.3f}) ===")
print(f"\nConfusion Matrix:")
print(f"              Predicted Low  Predicted High")
print(f"  Actual Low    {tn:>10,}    {fp:>10,}")
print(f"  Actual High   {fn:>10,}    {tp:>10,}")
print(f"\n{'Metric':<15} {'Value':>10}")
print("─" * 28)
print(f"{'Accuracy':<15} {acc:>10.4f}")
print(f"{'Precision':<15} {prec:>10.4f}")
print(f"{'Recall':<15} {rec:>10.4f}")
print(f"{'F1 Score':<15} {f1:>10.4f}")
print(f"{'True Positives':<15} {tp:>10,}")
print(f"{'False Positives':<15} {fp:>10,}")
print(f"{'True Negatives':<15} {tn:>10,}")
print(f"{'False Negatives':<15} {fn:>10,}")

# INTERPRETATION: Precision = "of those I predicted high, how many
# actually were?" Recall = "of those actually high, how many did I
# catch?" F1 balances both. In property, recall matters more if
# missing high-value flats is costly.



### Checkpoint 1


In [ ]:
assert tp + fp + tn + fn == n_obs, "Confusion matrix must sum to n"
assert 0 < prec <= 1, "Precision must be valid"
assert 0 < rec <= 1, "Recall must be valid"
print("\n[ok] Checkpoint 1 passed — confusion matrix + metrics computed\n")



## TASK 3 — TRAIN: ROC curve + precision-recall curve


In [ ]:
# ROC curve
fpr, tpr, roc_thresholds = roc_curve(y, p_scratch)
roc_auc = auc(fpr, tpr)

# Threshold closest to top-left corner (0, 1)
distances = np.sqrt((0 - fpr) ** 2 + (1 - tpr) ** 2)
roc_optimal_idx = np.argmin(distances)
roc_optimal_threshold = roc_thresholds[roc_optimal_idx]

print(f"\n=== ROC Curve ===")
print(f"AUC = {roc_auc:.4f}")
print(f"ROC-optimal threshold (closest to top-left): {roc_optimal_threshold:.4f}")
print(f"  at FPR={fpr[roc_optimal_idx]:.4f}, TPR={tpr[roc_optimal_idx]:.4f}")
print(f"\nAUC interpretation:")
if roc_auc > 0.9:
    print(f"  Excellent discrimination")
elif roc_auc > 0.8:
    print(f"  Good discrimination")
elif roc_auc > 0.7:
    print(f"  Fair discrimination")
else:
    print(f"  Poor discrimination")

# Precision-recall curve
prec_curve, rec_curve, pr_thresholds = precision_recall_curve(y, p_scratch)
pr_auc = auc(rec_curve, prec_curve)

print(f"\n=== Precision-Recall Curve ===")
print(f"PR-AUC = {pr_auc:.4f}")
print(f"Baseline (random): {n_positive/n_obs:.4f}")
print(f"PR-AUC / baseline: {pr_auc / (n_positive/n_obs):.2f}x better than random")

# INTERPRETATION: The PR curve is more informative than ROC when
# classes are imbalanced. For balanced classes (like ours), both
# tell a similar story. PR-AUC > baseline means the model does better
# than random guessing.



### Checkpoint 2


In [ ]:
assert 0.5 <= roc_auc <= 1.0, "AUC must be between 0.5 and 1.0"
assert pr_auc > n_positive / n_obs, "PR-AUC should beat random baseline"
print("\n[ok] Checkpoint 2 passed — ROC + PR curves computed\n")



## TASK 4 — VISUALISE: ROC, PR, and annotated confusion matrix


In [ ]:
# Plot 1: ROC curve
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=fpr, y=tpr, name=f"ROC (AUC={roc_auc:.3f})"))
fig1.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        name="Random",
        line={"dash": "dash", "color": "grey"},
    )
)
fig1.add_trace(
    go.Scatter(
        x=[fpr[roc_optimal_idx]],
        y=[tpr[roc_optimal_idx]],
        mode="markers",
        name=f"Optimal (t={roc_optimal_threshold:.3f})",
        marker={"size": 12, "color": "red"},
    )
)
fig1.update_layout(
    title="ROC Curve — HDB Resale Classification",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
)
fig1.write_html(str(OUTPUT_DIR / "roc_curve.html"))
print(f"Saved: {OUTPUT_DIR / 'roc_curve.html'}")

# Plot 2: Precision-recall curve
fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(
        x=rec_curve,
        y=prec_curve,
        name=f"PR (AUC={pr_auc:.3f})",
    )
)
fig2.add_hline(
    y=n_positive / n_obs,
    line_dash="dash",
    line_color="grey",
    annotation_text=f"Random baseline ({n_positive/n_obs:.2f})",
)
fig2.update_layout(
    title="Precision-Recall Curve — HDB Resale Classification",
    xaxis_title="Recall",
    yaxis_title="Precision",
)
fig2.write_html(str(OUTPUT_DIR / "precision_recall.html"))
print(f"Saved: {OUTPUT_DIR / 'precision_recall.html'}")

# Plot 3: Annotated confusion matrix heatmap
cm_labels = [["TN", "FP"], ["FN", "TP"]]
cm_text = [[f"{cm_labels[r][c]}\n{cm[r,c]:,}" for c in range(2)] for r in range(2)]

fig3 = go.Figure(
    data=go.Heatmap(
        z=cm,
        x=["Predicted Low", "Predicted High"],
        y=["Actual Low", "Actual High"],
        text=cm_text,
        texttemplate="%{text}",
        colorscale="Blues",
        showscale=False,
    )
)
fig3.update_layout(
    title=f"Confusion Matrix (t={optimal_threshold:.3f})",
    xaxis_title="Predicted",
    yaxis_title="Actual",
)
fig3.write_html(str(OUTPUT_DIR / "confusion_matrix.html"))
print(f"Saved: {OUTPUT_DIR / 'confusion_matrix.html'}")



### Checkpoint 3


In [ ]:
print("\n[ok] Checkpoint 3 passed — ROC, PR, confusion matrix visualised\n")



## TASK 5 — APPLY: bank mortgage pre-screening


Annual (RM waste, lost cross-sell) in S$, scaling dataset rates.


In [ ]:
# SCENARIO: A Singapore bank pre-screens HDB mortgage applications. Each
# application is classified as "likely high-value" or "likely standard."
#
# High-value applications are routed to a senior relationship manager
# (RM) who can offer premium rates and cross-sell wealth products.
# Standard applications go through the automated approval pipeline.
#
# Getting this wrong has two (ILLUSTRATIVE) costs:
#   - FP (standard routed to RM): RM time wasted, S$200/case
#   - FN (high-value goes auto): lost cross-sell revenue, S$8,000/case
#
# These are NOT the Task 2 costs (S$30K vs S$50K), so the Task 2
# threshold is not cost-optimal here — re-run the sweep with the
# bank's own costs before comparing thresholds.

rm_cost = 200  # S$ per unnecessary RM routing (illustrative)
cross_sell_loss = 8_000  # S$ lost revenue per missed high-value (illustrative)
annual_apps = 25_000  # illustrative annual application volume

bank_thresholds = np.round(np.linspace(0.01, 0.99, 99), 2)
bank_costs = []
for t in bank_thresholds:
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y, (p_scratch >= t).astype(int)).ravel()
    bank_costs.append(fp_t * rm_cost + fn_t * cross_sell_loss)
bank_threshold = float(bank_thresholds[int(np.argmin(bank_costs))])


def annual_bank_cost(threshold: float) -> tuple[float, float]:
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(
        y, (p_scratch >= threshold).astype(int)
    ).ravel()
    return (
        annual_apps * fp_t / n_obs * rm_cost,
        annual_apps * fn_t / n_obs * cross_sell_loss,
    )


print(f"\n=== Real-World Application: Bank Mortgage Pre-Screening ===")
print(f"  Annual mortgage applications (illustrative): {annual_apps:,}")
print(f"  Cost ratio FN:FP = {cross_sell_loss / rm_cost:.0f}:1")
bank_rows = [
    ("Default", 0.5),
    ("Task 2 optimum (other costs)", float(optimal_threshold)),
    ("Re-optimised for bank costs", bank_threshold),
]
print(f"\n  {'Threshold rule':<30} {'t':>5} {'RM waste S$':>13} {'Lost x-sell S$':>15} {'Total S$':>13}")
bank_totals = {}
for label, t in bank_rows:
    rm_waste, lost = annual_bank_cost(t)
    bank_totals[label] = rm_waste + lost
    print(f"  {label:<30} {t:>5.2f} {rm_waste:>13,.0f} {lost:>15,.0f} {rm_waste + lost:>13,.0f}")
savings = bank_totals["Default"] - bank_totals["Re-optimised for bank costs"]
print(f"\n  Annual saving vs default threshold: S${savings:,.0f}")
print(f"  Model discrimination (AUC): {roc_auc:.4f}")



### Checkpoint 4


In [ ]:
assert bank_totals["Re-optimised for bank costs"] <= min(
    bank_totals["Default"], bank_totals["Task 2 optimum (other costs)"]
), "The threshold tuned for the bank's costs must be the cheapest of the three"
print("\n[ok] Checkpoint 4 passed — threshold re-optimised for the bank's costs\n")

# BUSINESS IMPACT: The AUC printed above measures how well the model
# RANKS high-value above standard applications at every threshold; the
# threshold decides how that ranking becomes routing. With a 40:1 cost
# asymmetry the cheapest threshold sits below 0.5, sending more cases
# to RMs to miss fewer high-value clients. The table shows what reusing
# a threshold tuned for different costs would leave on the table.
#
# LIMITATIONS:
#   - Model uses only 3 property features. Real mortgage models
#     include income, existing debt, credit score, and employment.
#   - Cross-sell revenue is estimated. Actual conversion rates for
#     wealth products vary by customer segment and market conditions.
#   - The model does not account for seasonal effects (festive
#     periods, cooling measures) that shift the value distribution.



## REFLECTION


What you've mastered in this technique:
  ✓ Building a confusion matrix and deriving precision, recall, F1, accuracy
  ✓ ROC curve + AUC as threshold-free discrimination
  ✓ Precision-recall curve and when it beats ROC (rare positives)
  ✓ Choosing the operating threshold from the costs of the decision
    that will actually use it

Next: In 04_calibration_anova.py you'll check whether the predicted
probabilities themselves can be trusted (calibration) and compare
group means with ANOVA.
